# MVP 1단계 — 6종 가전 데이터셋 구축 & train/val/test 시간분할

> Jira: [S15P21D201-128] · 계획: `MVP실험계획.md` 1단계 · 작성: 2026-08-28

**입력 (선행 조건)**
- `ValidData/cache_1hz/*.npz` — 131채널 1Hz 캐시 (완비 상태)
- `ValidData/VL_extracted/` — 라벨 JSON 4,061개

**출력**
- `ValidData/dataset_mvp/` — 가구별 메인 X + 채널별 y (npz)
- `ValidData/dataset_mvp/split_manifest.json` — 분할 정의 + 채널 통계

**규격 (계획서 §2~3)** — 6종 가전 32채널 · Train 1~19일 / Val 20~24일 / Test 25~31일 · 다변량 4특징 · ON = 가전 채널 >50W · 인덕션은 10분 gap 선병합(조리 세션)

위에서부터 순서대로 전체 셀을 실행하세요. 마지막 셀이 **Claude 보고용 체크리스트**를 자동으로 채워 출력합니다.

In [1]:
# [1] 설정
import numpy as np, pandas as pd, json, glob, os, time
from collections import defaultdict

T_START = time.time()
BASE  = r"C:\Users\SSAFY\Documents\전기 인프라 지능화를 위한 가전기기 전력 사용량 데이터"
CACHE = os.path.join(BASE, "ValidData", "cache_1hz")
LBL   = os.path.join(BASE, "ValidData", "VL_extracted")
OUT   = os.path.join(BASE, "ValidData", "dataset_mvp")
os.makedirs(OUT, exist_ok=True)

# 6종 타겟 가전 (라벨 name 정확 일치)
TARGETS = ["전기포트", "인덕션(전기레인지)", "전기다리미", "전자레인지", "헤어드라이기", "진공 청소기(유선)"]
EXPECTED_CHANNELS = 32          # EDA 실측 기대값
ON_THRESH = 50.0                # W
INDUCTION_MERGE_S = 600         # 인덕션 조리 세션 선병합 gap
SPLIT = {"train": (0, 19), "val": (19, 24), "test": (24, 31)}   # 일 인덱스 [시작, 끝)
# 캐시 컬럼: [active, reactive, apparent, current, power_factor] → 학습 4특징 선택
FEAT_IDX = [0, 1, 4, 3]         # active, reactive, PF, current
FEAT_NAMES = ["active_power", "reactive_power", "power_factor", "current"]
print("설정 완료 |", OUT)

설정 완료 | C:\Users\SSAFY\Documents\전기 인프라 지능화를 위한 가전기기 전력 사용량 데이터\ValidData\dataset_mvp


In [2]:
# [2] 라벨 스캔 → 32채널 목록 생성 및 검증
channels = {}   # (house, ch) -> {name, dates}
for fp in glob.glob(os.path.join(LBL, "house_*", "ch*", "*.json")):
    with open(fp, encoding="utf-8") as f:
        d = json.load(f)
    m = d["meta"]
    if m["name"] not in TARGETS:
        continue
    h, ch = m["id"].split("_")
    k = (h, ch)
    channels.setdefault(k, {"name": m["name"], "dates": set()})
    channels[k]["dates"].add(m["date"])

houses = sorted({h for h, c in channels})
per_app = defaultdict(list)
for (h, c), v in channels.items():
    per_app[v["name"]].append(f"{h}/{c}")
print(f"타겟 채널: {len(channels)}개 (기대 {EXPECTED_CHANNELS}) · 가구 {len(houses)}곳")
for n, lst in sorted(per_app.items(), key=lambda x: -len(x[1])):
    print(f"  {n}: {len(lst)} -> {sorted(lst)}")
CHK_channel_count = len(channels)
assert CHK_channel_count == EXPECTED_CHANNELS, f"채널 수 불일치! {CHK_channel_count} != {EXPECTED_CHANNELS} — Claude에게 보고"
assert all(len(v["dates"]) == 31 for v in channels.values()), "31일 미만 채널 존재 — Claude에게 보고"

타겟 채널: 32개 (기대 32) · 가구 15곳
  전자레인지: 8 -> ['H022/ch09', 'H023/ch09', 'H026/ch09', 'H029/ch09', 'H057/ch09', 'H082/ch09', 'H083/ch09', 'H102/ch09']
  진공 청소기(유선): 6 -> ['H026/ch08', 'H069/ch08', 'H083/ch08', 'H102/ch08', 'H103/ch08', 'H105/ch08']
  전기다리미: 5 -> ['H022/ch18', 'H027/ch18', 'H070/ch18', 'H077/ch18', 'H083/ch18']
  인덕션(전기레인지): 5 -> ['H027/ch16', 'H069/ch16', 'H070/ch16', 'H077/ch16', 'H098/ch16']
  전기포트: 4 -> ['H029/ch04', 'H083/ch04', 'H098/ch04', 'H105/ch04']
  헤어드라이기: 4 -> ['H083/ch07', 'H098/ch07', 'H103/ch07', 'H105/ch07']


In [3]:
# [3] 분할 manifest 생성 (가구별 달력일 3분할, 자정 경계)
manifest = {"created": pd.Timestamp.now().isoformat(), "split_days": SPLIT,
            "feat_names": FEAT_NAMES, "on_thresh_w": ON_THRESH,
            "induction_merge_s": INDUCTION_MERGE_S, "houses": {}, "channels": {}}
for h in houses:
    dates = sorted(set().union(*[v["dates"] for (hh, c), v in channels.items() if hh == h]))
    assert len(dates) == 31, f"{h}: 날짜 수 {len(dates)} != 31"
    manifest["houses"][h] = {s: dates[a:b] for s, (a, b) in SPLIT.items()}
print("분할 예시", houses[0], {s: (v[0], v[-1]) for s, v in manifest["houses"][houses[0]].items()})

분할 예시 H022 {'train': ('20231030', '20231117'), 'val': ('20231118', '20231122'), 'test': ('20231123', '20231129')}


In [4]:
# [4] 캐시 무결성 검증 + 데이터셋 빌드
def close_mask(y, gap):   # 인덕션 선병합: gap초 이하 OFF 구간 메움 (모폴로지 closing)
    on = np.flatnonzero(y)
    if len(on) == 0: return y
    out = y.copy()
    d = np.diff(on)
    for i in np.flatnonzero((d > 1) & (d <= gap)):
        out[on[i]:on[i + 1]] = 1
    return out

gap_warn, val_test_zero, ch_stats = [], [], []
for h in houses:
    mz = np.load(os.path.join(CACHE, f"{h}_ch01.npz"), allow_pickle=True)
    t0 = pd.Timestamp(str(mz["t0"]))
    X = mz["x"][:, FEAT_IDX].astype(np.float32)
    n = len(X)
    gap_pct = float(np.isnan(X[:, 0]).mean() * 100)
    if gap_pct > 1: gap_warn.append((f"{h}_ch01", round(gap_pct, 2)))
    X = pd.DataFrame(X).ffill().bfill().values.astype(np.float32)
    dates_all = manifest["houses"][h]["train"] + manifest["houses"][h]["val"] + manifest["houses"][h]["test"]
    day_of = pd.Series((t0 + pd.to_timedelta(np.arange(n), unit="s")).strftime("%Y%m%d"))
    set_id = np.full(n, -1, dtype=np.int8)   # 0 train / 1 val / 2 test
    for si, s in enumerate(["train", "val", "test"]):
        set_id[np.isin(day_of.values, manifest["houses"][h][s])] = si
    np.savez_compressed(os.path.join(OUT, f"{h}_mains.npz"), X=X, set_id=set_id, t0=str(t0))
    for (hh, c), v in sorted(channels.items()):
        if hh != h: continue
        cz = np.load(os.path.join(CACHE, f"{h}_{c}.npz"), allow_pickle=True)
        ct0 = pd.Timestamp(str(cz["t0"]))
        ap = cz["x"][:, 0]
        gp = float(np.isnan(ap).mean() * 100)
        if gp > 1: gap_warn.append((f"{h}_{c}", round(gp, 2)))
        off = int((ct0 - t0).total_seconds())
        y = np.zeros(n, dtype=np.uint8)
        src = np.nan_to_num(ap) > ON_THRESH
        s0, s1 = max(0, -off), min(len(src), n - off)
        y[s0 + off:s1 + off] = src[s0:s1]
        if v["name"] == "인덕션(전기레인지)":
            y = close_mask(y, INDUCTION_MERGE_S)
        np.savez_compressed(os.path.join(OUT, f"{h}_{c}_y.npz"), y=y)
        row = {"key": f"{h}_{c}", "name": v["name"]}
        for si, s in enumerate(["train", "val", "test"]):
            m = set_id == si
            ys = y[m]
            ev = int(np.sum(np.diff(np.concatenate([[0], ys])) == 1))
            row[f"{s}_on_pct"] = round(float(ys.mean()) * 100, 3)
            row[f"{s}_events"] = ev
            if s in ("val", "test") and ev == 0: val_test_zero.append((f"{h}_{c}", v["name"], s))
        ch_stats.append(row)
        manifest["channels"][f"{h}_{c}"] = row
    print(f"{h} 완료 ({time.time()-T_START:.0f}s)", flush=True)

with open(os.path.join(OUT, "split_manifest.json"), "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=1)
print("\nsplit_manifest.json 저장 완료")

H022 완료 (1431s)
H023 완료 (1443s)
H026 완료 (1456s)
H027 완료 (1470s)
H029 완료 (1483s)
H057 완료 (1495s)
H069 완료 (1509s)
H070 완료 (1521s)
H077 완료 (1536s)
H082 완료 (1551s)
H083 완료 (1567s)
H098 완료 (1582s)
H102 완료 (1597s)
H103 완료 (1610s)
H105 완료 (1623s)

split_manifest.json 저장 완료


In [5]:
# [5] 세트별 통계 요약표
df = pd.DataFrame(ch_stats).set_index(["name", "key"]).sort_index()
display(df)
print("\n가전별 이벤트 합계 (train / val / test):")
agg = df.groupby("name")[["train_events", "val_events", "test_events"]].sum()
display(agg)

train_on_pct  train_events  val_on_pct  val_events  \
name       key                                                             
인덕션(전기레인지) H027_ch16         0.946            21       1.454          11   
           H069_ch16         7.223           123       7.227          35   
           H070_ch16         2.631            33       2.914          10   
           H077_ch16         0.886            14       1.247           5   
           H098_ch16         1.469            32       1.750           9   
전기다리미      H022_ch18         0.131           126       0.400          89   
           H027_ch18         0.181           153       0.248          38   
           H070_ch18         0.229           281       0.228          74   
           H077_ch18         0.223           145       0.277          46   
           H083_ch18         0.236           221       0.214          56   
전기포트       H029_ch04         0.413            32       0.292           6   
           H083_ch04         0.422            59       0.348           8   
           H098_ch04         0.655            53       0.587          16   
           H105_ch04         0.602            89       0.600          22   
전자레인지      H022_ch09         0.302            25       0.411           8   
           H023_ch09         0.114            95       0.073          22   
           H026_ch09         0.283            22       0.585          11   
           H029_ch09         0.294            13       0.554           4   
           H057_ch09         0.262            43       0.213          32   
           H082_ch09         0.378            46       0.455          13   
           H083_ch09         0.743           114       0.525          22   
           H102_ch09         0.704            72       0.531          13   
진공 청소기(유선) H026_ch08         0.242            22       0.523           6   
           H069_ch08         0.298            74       0.423          22   
           H083_ch08         0.418            43       0.509          14   
           H102_ch08         0.472            77       0.719          14   
           H103_ch08         0.506            68       0.627          19   
           H105_ch08         0.334           118       0.361          41   
헤어드라이기     H083_ch07         0.497            80       0.473          16   
           H098_ch07         0.490            88       0.443          12   
           H103_ch07         0.490            65       0.588          19   
           H105_ch07         0.408            67       0.443          15   

                      test_on_pct  test_events  
name       key                                  
인덕션(전기레인지) H027_ch16        0.967           10  
           H069_ch16        6.014           37  
           H070_ch16        3.066           16  
           H077_ch16        0.772            7  
           H098_ch16        1.141           13  
전기다리미      H022_ch18        0.037           18  
           H027_ch18        0.254           51  
           H070_ch18        0.221           98  
           H077_ch18        0.248           59  
           H083_ch18        0.244           94  
전기포트       H029_ch04        0.133            4  
           H083_ch04        0.379           22  
           H098_ch04        0.611           25  
           H105_ch04        0.611           36  
전자레인지      H022_ch09        0.074            5  
           H023_ch09        0.228            4  
           H026_ch09        0.438           10  
           H029_ch09        0.138            2  
           H057_ch09        0.120            8  
           H082_ch09        0.484           27  
           H083_ch09        0.714           57  
           H102_ch09        0.607           23  
진공 청소기(유선) H026_ch08        0.411            7  
           H069_ch08        0.478           34  
           H083_ch08        0.443           30  
           H102_ch08        0.685           29  
           H103_ch08        0.478           36  
           H105_


가전별 이벤트 합계 (train / val / test):


,train_events,val_events,test_events
name,,,
인덕션(전기레인지),223,70,83
전기다리미,926,303,320
전기포트,233,52,87
전자레인지,430,125,136
진공 청소기(유선),402,116,190
헤어드라이기,300,62,75


In [6]:
# [6] ✅ Claude 보고용 체크리스트 (자동 채움) — 아래 출력 전체를 복사해서 전달
size_gb = sum(os.path.getsize(os.path.join(OUT, f)) for f in os.listdir(OUT)) / 1e9
elapsed = (time.time() - T_START) / 60
tot = df[["train_events", "val_events", "test_events"]].sum()
print("=" * 62)
print("MVP 1단계 완료 보고 (이 블록을 Claude에게 붙여넣기)")
print("=" * 62)
print(f"[1] 전체 셀 오류 없이 완료: 예 (소요 {elapsed:.1f}분)")
print(f"[2] 타겟 채널 수: {CHK_channel_count} / 기대 {EXPECTED_CHANNELS}")
print(f"[3] 산출물: {OUT}")
print(f"    파일 {len(os.listdir(OUT))}개, {size_gb:.2f} GB (mains 15 + y 32 + manifest)")
print(f"[4] 이벤트 합계: train {tot['train_events']} / val {tot['val_events']} / test {tot['test_events']}")
print(f"[5] Val/Test 이벤트 0건 채널 ({len(val_test_zero)}건): {val_test_zero if val_test_zero else '없음'}")
print(f"[6] 결측 1% 초과 채널 ({len(gap_warn)}건): {gap_warn if gap_warn else '없음'}")
print(f"[7] 특이사항/에러 메시지: (직접 기입, 없으면 '없음')")
print("=" * 62)
print("→ [5]에 항목이 있으면 해당 채널의 평가 방법을 Claude와 조정 필요")
print("→ 이상 없으면 2단계(32채널 학습) 진행 가능")

MVP 1단계 완료 보고 (이 블록을 Claude에게 붙여넣기)
[1] 전체 셀 오류 없이 완료: 예 (소요 28.2분)
[2] 타겟 채널 수: 32 / 기대 32
[3] 산출물: C:\Users\SSAFY\Documents\전기 인프라 지능화를 위한 가전기기 전력 사용량 데이터\ValidData\dataset_mvp
    파일 48개, 0.46 GB (mains 15 + y 32 + manifest)
[4] 이벤트 합계: train 2514 / val 728 / test 891
[5] Val/Test 이벤트 0건 채널 (0건): 없음
[6] 결측 1% 초과 채널 (0건): 없음
[7] 특이사항/에러 메시지: (직접 기입, 없으면 '없음')
→ [5]에 항목이 있으면 해당 채널의 평가 방법을 Claude와 조정 필요
→ 이상 없으면 2단계(32채널 학습) 진행 가능


## 완료 후 Claude에게 알려줄 체크리스트

위 [6] 셀의 출력 블록을 그대로 복사해 전달하면 됩니다. 수동 확인 항목:

- [ ] **[1]** 모든 셀이 위에서부터 오류 없이 실행됐는가 (assert 실패·예외 발생 시 그 메시지 전체를 전달)
- [ ] **[2]** 채널 수 32 일치 여부
- [ ] **[3]** `dataset_mvp` 폴더 생성 및 용량
- [ ] **[4]** 세트별 이벤트 합계 — val·test가 0에 가깝게 비어 있지 않은지
- [ ] **[5]** Val/Test 이벤트 0건 채널 목록 — **있으면 반드시 전달** (해당 채널은 임계값 튜닝·평가 방식 조정 필요)
- [ ] **[6]** 결측 1% 초과 채널 — 있으면 전달
- [ ] **[7]** 실행 중 특이사항 (메모리 부족, 커널 재시작, 경고 등)

**보고를 받으면 Claude가 하는 일**: [5]·[6] 항목 채널의 처리 방침 결정 → 2단계(32채널 TCN & Stats-MLP 학습, S15P21D201-143) 스크립트를 이 manifest 기준으로 실행.